# 00 Setup and data: meet Meridian

Meridian is a fictional travel company. Customers write in; the router has to send each message to one of eight queues.
This notebook checks the environment, downloads the three encoders the workshop uses, and loads the triage set.

**Provenance of the data.** The 1,200 messages were written by an LLM (Claude Haiku) from route definitions. Every label is the
generator's intended route and **no person has read the test split yet**, so treat the test numbers in later notebooks as
a measurement on synthetic data, not as a claim about real customers.

In [1]:
import platform, sys, os
import numpy as np
import torch, transformers, sklearn
import meridian as m

print("python", sys.version.split()[0], "| torch", torch.__version__, "| transformers", transformers.__version__, "| sklearn", sklearn.__version__)
print("hardware:", m.hardware())
torch.set_num_threads(os.cpu_count())
print("torch threads:", torch.get_num_threads())

/Users/ben/deterministic-ai-workshop/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


python 3.11.9 | torch 2.2.2 | transformers 4.57.6 | sklearn 1.8.0
hardware: Intel(R) Core(TM) i7-7567U CPU @ 3.50GHz, 4 logical cores, Darwin 22.6.0
torch threads: 4


## The data: counts per split and route
Splits are by scenario: the five variants of one situation never straddle splits.

In [2]:
import collections
rows = m.load()
by_split = collections.Counter(r["split"] for r in rows)
print(dict(by_split))
for split in ("train", "calibration", "test"):
    c = collections.Counter(r["label"] for r in m.load(split))
    print(f"{split:12s}", {k: c[k] for k in m.ROUTES})

{'train': 800, 'calibration': 200, 'test': 200}
train        {'baggage_claim': 100, 'cancellation_request': 100, 'out_of_scope': 100, 'payment_dispute': 100, 'rebooking_request': 100, 'refund_claim': 100, 'service_complaint': 100, 'visa_entry_question': 100}
calibration  {'baggage_claim': 25, 'cancellation_request': 25, 'out_of_scope': 25, 'payment_dispute': 25, 'rebooking_request': 25, 'refund_claim': 25, 'service_complaint': 25, 'visa_entry_question': 25}
test         {'baggage_claim': 25, 'cancellation_request': 25, 'out_of_scope': 25, 'payment_dispute': 25, 'rebooking_request': 25, 'refund_claim': 25, 'service_complaint': 25, 'visa_entry_question': 25}


## What a message looks like
One easy message and one boundary case (a message that mentions a neighboring route).

In [3]:
easy = next(r for r in rows if not r["boundary"] and r["label"] == "refund_claim")
hard = next(r for r in rows if r["boundary"] and r["label"] == "refund_claim")
for r in (easy, hard):
    print(r["label"], "| boundary:", r["boundary"], "| persona:", r["persona"])
    print("   ", r["text"], "\n")

lengths = np.array([len(r["text"].split()) for r in rows])
print(f"words per message: min {lengths.min()}, median {int(np.median(lengths))}, max {lengths.max()}")

refund_claim | boundary: False | persona: rushed
    hey my flight to lisbon MER-48213 got cancelled by the airline last nite and i was rebooked for thursday, cant do thurs, need money back for the unused ticket pls asap 

refund_claim | boundary: True | persona: rushed
    athens flight MER-15392 cancelled by airline, they charged me 25 eur to rebook which is ridiculous. i want the full fare back since they cancelled, not a rebooking 

words per message: min 8, median 27, max 51


## Download the encoders
Three models, all small enough for a laptop CPU. The first run downloads about 1.3 GB; later runs read the cache.
`bge-small-en-v1.5` (33M parameters) is a sentence-embedding model and the strong small baseline.
ModernBERT-base (149M) and LFM2.5-Encoder-230M are general encoders; later notebooks measure what they give when frozen.

In [4]:
for name, cfg in m.ENCODERS.items():
    try:
        tok, mod = m.load_encoder(name)
        n = sum(p.numel() for p in mod.parameters()) / 1e6
        print(f"{name:11s} {cfg['repo']:34s} {n:6.1f}M parameters, pooling: {cfg['pool']}")
        del mod
    except Exception as e:
        print(f"{name:11s} {cfg['repo']:34s} FAILED: {type(e).__name__}: {str(e)[:120]}")

bge-small   BAAI/bge-small-en-v1.5               33.4M parameters, pooling: cls


modernbert  answerdotai/ModernBERT-base         149.0M parameters, pooling: mean


lfm2.5-enc  LiquidAI/LFM2.5-Encoder-230M        229.7M parameters, pooling: mean


## A silent failure to check for: weights that never loaded
The LFM2.5-Encoder model card says to load the encoder body with `AutoModel`. With `transformers` 4.57.6 that call prints a warning that
every layer was "newly initialized", because the checkpoint stores the weights under an `lfm2.` prefix that `AutoModel` does not look for.
The model then runs without error and returns embeddings from random weights. `meridian.load_encoder` loads the masked-language-model class and
takes its `lfm2` body instead. The cell below shows the check that catches the difference: a loaded encoder fills in a masked word.

In [5]:
import torch
from transformers import AutoModelForMaskedLM, AutoTokenizer
repo = m.ENCODERS["lfm2.5-enc"]["repo"]
tok = AutoTokenizer.from_pretrained(repo, trust_remote_code=True)
mlm = AutoModelForMaskedLM.from_pretrained(repo, trust_remote_code=True).eval()
t = tok(f"The capital of France is {tok.mask_token}.", return_tensors="pt")
with torch.inference_mode():
    logits = mlm(**t).logits
i = (t["input_ids"][0] == tok.mask_token_id).nonzero()[0].item()
print("masked-word top 3:", [tok.decode(x).strip() for x in logits[0, i].topk(3).indices])

masked-word top 3: ['Paris', 'Strasbourg', 'Paris']
